# 11 — Machine Learning untuk Kolorimetri Citra

### Dari pengukuran RGB ke keputusan model yang tahan terhadap perubahan pencahayaan dan perangkat

**Seri:** Pengolahan Citra Digital dan *Scientific Imaging* · **Bahasa:** Indonesia · **Prasyarat:** Notebook 01–09, terutama 07 (warna) dan 09 (evaluasi klasifikasi). **Durasi yang disarankan:** 2–3 pertemuan atau praktikum mandiri bertahap.

Pada notebook 07, kita membahas bagaimana respons sensor dan transformasi warna memengaruhi *fidelitas warna*. Pada notebook 09, kita mempelajari bahwa model klasifikasi harus dinilai melalui data uji terpisah. Notebook 11 mempertemukan keduanya melalui persoalan teknik: **apakah model yang mempelajari warna permukaan tetap dapat dipercaya ketika iluminan, gain, dan karakteristik perangkat berubah?**

Kita akan membangun sistem citra dua bagian: sampel berwarna dan *patch* netral yang difoto secara bersamaan. Nilai RGB mentah dibandingkan dengan *chromaticity*, *white balance* berbasis referensi, serta transformasi warna yang dilatih pada chart terpisah. Setelah itu, beberapa model diperbandingkan dengan prosedur *train–validation–test* berdasarkan sesi akuisisi. Seluruh eksperimen berbasis **data dan perangkat sintetis**; hasil bukan kalibrasi kamera fisik, bukan pembacaan spektrofotometer, dan bukan validasi mutu produk di lapangan.

**Pertanyaan sentral:** kapan koreksi warna membantu prediksi, kapan justru memperbesar noise, dan mengapa perbaikan $\Delta E_{00}$ tidak selalu sejalan dengan peningkatan F1-score?

## Peta kompetensi dan diagnosis awal

Setelah menyelesaikan materi, pembaca diharapkan mampu:

1. Menghubungkan reflektansi, SPD iluminan, respons sensor, *camera RGB*, dan representasi warna digital.
2. Menjelaskan mengapa *white balance* diagonal berbeda dari *color correction matrix* (CCM) dan kalibrasi kolorimetrik penuh.
3. Menghitung dan menafsirkan *chromaticity*, CIELAB, serta $\Delta E_{00}$ dalam kerangka referensi yang dinyatakan jelas.
4. Menurunkan solusi *ridge regression* untuk pemetaan warna multioutput dan menjelaskan alasan regularisasi.
5. Mengembangkan fitur citra yang memperhatikan pencahayaan, referensi netral, dan karakter noise.
6. Merancang eksperimen klasifikasi dengan *grouped split* dan pemilihan hiperparameter yang tidak menggunakan data test.
7. Membandingkan akurasi warna, macro-F1, kalibrasi probabilitas, stabilitas lintas batch, dan biaya komputasi.
8. Mengukur penurunan kinerja pada kamera sintetis tak terlihat, serta ketidakpastian hasil dengan *bootstrap* pada unit batch.
9. Menyatakan keterbatasan simulasi dan merumuskan protokol validasi menggunakan perangkat fisik.

**Pertanyaan diagnostik:** Dua permukaan menghasilkan RGB yang sama pada kamera A tetapi berbeda pada kamera B. Apakah keduanya pasti memiliki reflektansi spektral yang sama? Jika memakai kartu abu-abu untuk koreksi kanal, dapatkah semua variasi warna antarperangkat dihilangkan? Mengapa 20 gambar dari satu sesi bukan otomatis 20 pengamatan statistik yang independen?

## 1. Kontrak numerik, perangkat lunak, dan reproduksibilitas

Citra yang dipakai berupa array **RGB linear sintetis** dengan nilai dalam $[0,1]$, bukan berkas JPEG atau RAW kamera nyata. Nilai yang dipakai untuk model sensor dibuat sebelum pengkodean gamma. Hanya untuk visualisasi dan konversi ke Lab, nilai linear *referensi sRGB* dikodekan dengan fungsi transfer sRGB. Seluruh operasi komputasi memakai floating point; tampilan `imshow` bukan bukti akurasi kolorimetrik monitor.

Satu **batch** adalah satu sesi akuisisi dengan iluminan dan karakter sensor yang sama. Setiap batch memuat seluruh kelas; pembagian data dilakukan pada unit batch dan ditetapkan sekali. Data *test* tidak digunakan untuk melatih model, mempelajari matriks warna, atau memilih hiperparameter. Dataset bisa dibangkitkan mandiri tanpa API, kamera, atau internet; paket yang diperlukan adalah NumPy, pandas, Matplotlib, scikit-image, SciPy, dan scikit-learn.

In [ ]:
from pathlib import Path
import platform, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats
from skimage.color import rgb2lab, deltaE_ciede2000
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import SVC
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import (accuracy_score, f1_score, balanced_accuracy_score,
    confusion_matrix, ConfusionMatrixDisplay, classification_report, log_loss)
import sklearn, skimage, scipy

SEED=20261112
rng=np.random.default_rng(SEED)
plt.rcParams.update({'figure.figsize':(9.3,4.3),'figure.dpi':105})
OUTPUT_DIR=Path('output_notebook_11')
OUTPUT_DIR.mkdir(exist_ok=True)
print('Python',platform.python_version(),'NumPy',np.__version__,'scikit-image',skimage.__version__,
      'scikit-learn',sklearn.__version__,'SciPy',scipy.__version__)

## 2. Fisika pengukuran warna: dari spektrum ke kanal sensor

Untuk permukaan dengan reflektansi spektral $R(\lambda)$, iluminan $E(\lambda)$, fungsi sensitivitas kanal kamera $s_k(\lambda)$, dan konstanta skala paparan $a$, respons ideal kanal $k$ dapat ditulis sebagai

$$
q_k=a\int_{\lambda_{\min}}^{\lambda_{\max}} E(\lambda)R(\lambda)s_k(\lambda)\,d\lambda + d_k + \eta_k,\qquad k\in\{R,G,B\}.
$$

Di sini $d_k$ adalah sinyal gelap/offset dan $\eta_k$ memodelkan noise. Respons sensor tidak harus identik dengan fungsi pencocokan warna CIE. Karena hasil integrasi spektrum yang sangat panjang diringkas menjadi tiga kanal, pemetaan balik ke spektrum umumnya **tidak unik**; kondisi metamerisme dan *camera metamerism* perlu diingat. Keseimbangan putih hanya dapat menghilangkan sebagian variasi, dan perubahan iluminan dapat menimbulkan efek non-diagonal yang tidak dapat diperbaiki dengan mengalikan tiga kanal secara terpisah.

Untuk simulator yang dapat diaudit, integral spektral ini kita ganti dengan aproksimasi tiga kanal:

$$
\mathbf y=\operatorname{clip}\!\left[a\,\operatorname{diag}(\boldsymbol\ell)\,\mathbf M_d\mathbf r+\mathbf b+\boldsymbol\eta,\,0,1\right],
$$

$d$ menandai perangkat, $\boldsymbol\ell$ gain iluminan, $\mathbf M_d$ matriks pencampuran kanal yang menyederhanakan respons sensor, $\mathbf r$ warna referensi linear sintetis, dan $\mathbf b$ offset. Model tiga kanal **bukan pengganti integrasi spektral** dan tidak mengandung geometri reflektansi kompleks; ia ditujukan untuk memisahkan faktor penyebab dan menguji sensitivitas model.

## 3. Apa yang disebut RGB linear dan sRGB?

Bilangan 0,5 pada berkas sRGB **tidak** menyatakan intensitas cahaya linear 50%. Sebelum menggunakan model pencahayaan aditif/multiplikatif dan pencampuran linier, kanal hasil pengkodean sRGB harus didekode. Hubungan *forward* untuk kanal linear $C_L$ adalah

$$
C_{\mathrm{sRGB}}=\begin{cases}
12.92 C_L,& C_L\leq0.0031308,\\
1.055 C_L^{1/2.4}-0.055,& C_L>0.0031308.
\end{cases}
$$

Fungsi ini didefinisikan dalam konteks warna sRGB dan tidak boleh langsung diterapkan pada RAW sensor yang belum dikarakterisasi. Pada simulator, $\mathbf r$ **didefinisikan sebagai koordinat linear sRGB referensi** sehingga ia dapat dikodekan menjadi sRGB dan dikonversi ke CIELAB untuk mengukur jarak dari referensi simulasi. Sebaliknya, RGB sensor sebelum CCM tidak memiliki interpretasi CIE yang sama. Jika nilainya dipaksa ke transformasi sRGB→Lab, nilai $\Delta E_{00}$ yang muncul hanyalah *proxy kesalahan terhadap target referensi*, bukan hasil kolorimetri kamera tersertifikasi.

In [ ]:
def srgb_encode(x):
    x=np.asarray(x,dtype=float)
    x=np.clip(x,0,1)
    return np.where(x<=0.0031308,12.92*x,1.055*np.power(x,1/2.4)-0.055)

def srgb_decode(x):
    x=np.asarray(x,dtype=float)
    return np.where(x<=0.04045,x/12.92,np.power((x+0.055)/1.055,2.4))

u=np.linspace(0,1,300)
fig,ax=plt.subplots(1,2,figsize=(11,3.7))
ax[0].plot(u,srgb_encode(u),label='pengkodean sRGB')
ax[0].plot(u,u,'--',label='identitas linear')
ax[0].set(xlabel='Intensitas linear (fraksi)',ylabel='Nilai sRGB',title='Fungsi transfer sRGB')
ax[0].legend();ax[0].grid(alpha=.2)
ax[1].plot(u,srgb_decode(srgb_encode(u))-u)
ax[1].set(xlabel='Intensitas linear',ylabel='Selisih setelah round-trip',title='Konsistensi numerik')
ax[1].grid(alpha=.2);plt.tight_layout();plt.show()
assert np.max(np.abs(srgb_decode(srgb_encode(u))-u))<1e-12

## 4. Masalah keputusan: kesamaan warna, label kelas, dan tujuan evaluasi

Menilai akurasi warna dan menilai akurasi klasifikasi adalah dua tugas berbeda. Tugas pertama menggunakan **referensi warna kontinu**, misalnya jarak $\Delta E_{00}$ terhadap warna target; tugas kedua menggunakan label kategori $y\in\{0,1,2,3\}$. Dua sampel bisa memiliki beda warna yang terukur namun termasuk kelas yang sama, atau perubahan kecil dapat menyeberangi batas keputusan classifier. Karena itu, mengurangi $\Delta E_{00}$ tidak menjamin F1 meningkat, dan macro-F1 tinggi tidak membuktikan bahwa kamera terkalibrasi.

Kita menggunakan empat kategori permukaan warna sintetis yang sengaja dibuat cukup berdekatan. Tiap kelas memiliki variasi antarsampel sehingga tugas tidak sekadar mencocokkan empat kode RGB persis. Keempat label adalah definisi dalam simulasi—**bukan diagnosis mutu, kesegaran, atau jenis material aktual**.

In [ ]:
NAMA_KELAS=np.array(['A — merah kusam','B — jingga redup','C — hijau kusam','D — ungu redup'])
PALET=np.array([
    [0.57,0.25,0.19],
    [0.47,0.35,0.21],
    [0.34,0.42,0.26],
    [0.35,0.30,0.43],
],dtype=float)
fig,ax=plt.subplots(figsize=(9,2.2))
ax.imshow(srgb_encode(PALET)[None,:,:],aspect='auto',extent=(0,4,0,1))
ax.set(xlim=(0,4),ylim=(0,1),xticks=np.arange(4)+.5,xticklabels=NAMA_KELAS,yticks=[],title='Warna pusat empat kelas (referensi sintetis)')
plt.show()
print('Selisih warna antar-pusat adalah informasi simulasi; tidak identik dengan toleransi material nyata.')

## 5. Desain eksperimen: sesi akuisisi, perangkat, dan warna referensi

Satu frame berisi objek di kiri dan *patch* netral di kanan. Satu batch mempunyai matriks respons perangkat dan gain iluminan yang sama, tetapi antarframe terdapat perubahan paparan, *shading* spasial, variasi bahan, dan noise. *Patch* netral berfungsi sebagai referensi yang diukur bersama objek; ia **bukan** ColorChecker dan tidak cukup untuk mengoreksi pencampuran spektral antar-kanal. Untuk menentukan CCM, kita menyediakan pengamatan chart warna sintetis yang referensinya diketahui dan yang diukur hanya pada batch yang diizinkan.

Terdapat dua skenario:

- **Domain utama:** perangkat sintetis A, variasi gain iluminan sedang, 36 batch, 20 frame per batch. Bagi batch ke train (24), validation (6), test (6).
- **Domain stress:** perangkat sintetis B yang belum terlihat dan variasi iluminan lebih kuat, 6 batch yang disimulasikan secara terpisah. Ini adalah pengujian domain berbeda, **bukan** sampel yang dapat langsung dianggap mewakili seluruh kamera nyata.

Setiap batch dibuat seimbang, sehingga hubungan kelas dengan identitas batch tidak sengaja menjadi pembeda. Nilai acak dan generator terdokumentasi; tidak ada gambar nyata yang didistribusikan.

In [ ]:
M_A=np.array([[1.00,0.065,-0.025],[-0.030,0.985,0.060],[0.045,-0.020,1.010]])
M_B=np.array([[0.83,0.18,0.045],[0.075,0.87,0.080],[0.17,0.070,0.80]])
NEUTRAL=0.42
CHART=np.array([
    [0.12,0.12,0.12],[0.25,0.25,0.25],[0.42,0.42,0.42],[0.70,0.70,0.70],
    [0.64,0.23,0.16],[0.52,0.43,0.20],[0.20,0.44,0.26],[0.22,0.27,0.64],
    [0.60,0.18,0.50],[0.17,0.56,0.52],[0.45,0.28,0.16],[0.25,0.34,0.50],
    [0.70,0.48,0.42],[0.38,0.52,0.27],[0.49,0.26,0.45],[0.29,0.53,0.54],
    [0.29,0.20,0.13],[0.78,0.63,0.32],[0.17,0.23,0.33],[0.60,0.59,0.57],
    [0.40,0.14,0.20],[0.12,0.39,0.48],[0.50,0.48,0.19],[0.40,0.39,0.40]
])

def hasil_sensor(ref, M, gains, exposure, offset, rng, sigma=.009):
    """Pendekatan sensor tiga kanal sintetis, respons terhadap reflektansi linear referensi."""
    x=np.asarray(ref,dtype=float)
    val=exposure*((x@M.T)*gains)+offset
    # Noise gabungan sederhana; bukan sampel shot-noise foton fisik terkalibrasi
    eta=rng.normal(0,sigma*np.sqrt(np.maximum(val,0)+.05),size=val.shape)
    return np.clip(val+eta,0,1)

def buat_dataset(batch_ids, seed, device='A', stress=False, frame_shape=(24,48)):
    """Simulator instruksional; satu frame = sampel kiri + referensi netral kanan."""
    gen=np.random.default_rng(seed)
    M=M_A if device=='A' else M_B
    h,w=frame_shape
    X=[]; y=[]; groups=[]; refs=[]; exposures=[]; charts=[]; chart_target=[]; patch_gain=[]
    shift_scale=.21 if not stress else .42
    for batch in batch_ids:
        illum=np.exp(gen.normal(0,shift_scale,3))
        illum=np.clip(illum,.43,1.80)
        off=gen.uniform(-.012,.012,3)
        shadow=gen.uniform(-.08,.08)
        cb=[]; tb=[]
        for target in CHART:
            ob=hasil_sensor(target,M,illum,1.0,off,gen,sigma=.006)
            neutral=hasil_sensor([NEUTRAL]*3,M,illum,1.0,off,gen,sigma=.004)
            cb.append((ob,neutral))
            tb.append(target)
        charts.extend(cb); chart_target.extend(tb)
        for label in range(4):
            for rep in range(5):
                target=np.clip(PALET[label]+gen.normal(0,.039,3),.04,.90)
                exposure=gen.uniform(.93,1.07)
                # Shading spasial: kiri dan kanan tidak benar-benar identik
                Y,Xg=np.mgrid[0:h,0:w]
                gradient=1+shadow*(Xg-(w-1)/2)/(w/2)+.025*np.sin(Y/6+batch)
                obj=hasil_sensor(target,M,illum,exposure,off,gen,sigma=.021)
                ref=hasil_sensor([NEUTRAL]*3,M,illum,exposure,off,gen,sigma=.021)
                frame=np.empty((h,w,3),dtype=np.float32)
                frame[:,:w//2,:]=np.clip(obj[None,None,:]*gradient[:,:w//2,None]+gen.normal(0,.010,(h,w//2,3)),0,1)
                frame[:,w//2:,:]=np.clip(ref[None,None,:]*gradient[:,w//2:,None]+gen.normal(0,.010,(h,w-w//2,3)),0,1)
                X.append(frame);y.append(label);groups.append(batch);refs.append(target)
                exposures.append(exposure);patch_gain.append(illum)
    return {'images':np.stack(X),'y':np.array(y,dtype=int),
            'batch':np.array(groups,dtype=int),'target':np.stack(refs),
            'chart_obs':np.stack([a for a,b in charts]),
            'chart_ref':np.stack([b for a,b in charts]),
            'chart_target':np.stack(chart_target),
            'chart_batch':np.repeat(batch_ids,len(CHART)),
            'device':device,'stress':stress,'illums':np.stack(patch_gain)}

domain=buat_dataset(np.arange(36),SEED+1,'A')
print('Citra:',domain['images'].shape,'| chart:',domain['chart_obs'].shape,
      '| perangkat:',domain['device'],'| kelas:',np.bincount(domain['y']))

## 6. Pengamatan visual: warna intrinsik dan hasil kamera tidak identik

Gambar berikut adalah hasil render simulator. Jika dua citra menampilkan permukaan kelas sama dengan tampilan berbeda, itu belum berarti material telah berubah; perbedaan bisa berasal dari gain iluminan, karakteristik respons sensor, paparan, dan pengkodean tampilan. Sebaliknya, tampilan layar yang tampak sama bukan jaminan koordinat warna atau reflektansi sama. Patch kanan adalah **referensi netral terukur**, bukan target warna yang memiliki informasi tentang kelas.

In [ ]:
fig,ax=plt.subplots(2,4,figsize=(11,5.4))
for j,b in enumerate([0,8,17,35]):
    ix=np.flatnonzero((domain['batch']==b)&(domain['y']==1))[0]
    ax[0,j].imshow(srgb_encode(domain['images'][ix]));ax[0,j].set_title(f'Kelas B, batch {b}')
    ax[0,j].axis('off')
    iy=np.flatnonzero((domain['batch']==b)&(domain['y']==3))[0]
    ax[1,j].imshow(srgb_encode(domain['images'][iy]));ax[1,j].set_title(f'Kelas D, batch {b}')
    ax[1,j].axis('off')
plt.suptitle('Objek di kiri dan patch netral di kanan; tampilan sintetis',y=1.02)
plt.tight_layout();plt.show()

## 7. Eksplorasi data dan pembagian berdasarkan batch

Pembagian citra acak **per frame** tidak sesuai apabila frame dari satu sesi berbagi konfigurasi sensor atau iluminasi yang sama. Ketergantungan dalam satu batch dapat membuat variasi dalam test menjadi tampak lebih kecil daripada variasi kondisi akuisisi baru. Kita membagi seluruh 36 batch menjadi 24/6/6 dengan **GroupShuffleSplit** yang deterministik. Hanya setelah pembagian ini fitur, CCM, model, dan aturan keputusan dipelajari. Satu perangkat B disimulasikan terpisah sebagai domain uji tekanan.

In [ ]:
ids=np.arange(len(domain['y']))
splitter=GroupShuffleSplit(n_splits=1,test_size=12/36,random_state=SEED)
train_idx,remain_idx=next(splitter.split(ids,domain['y'],domain['batch']))
second=GroupShuffleSplit(n_splits=1,test_size=.5,random_state=SEED+1)
val_rel,test_rel=next(second.split(remain_idx,domain['y'][remain_idx],domain['batch'][remain_idx]))
val_idx=remain_idx[val_rel];test_idx=remain_idx[test_rel]
sets={'train':train_idx,'validation':val_idx,'test':test_idx}
for key,ind in sets.items():
    print(key, 'citra',len(ind),'batch',len(np.unique(domain['batch'][ind])),
          'per kelas',np.bincount(domain['y'][ind],minlength=4))
assert len(set(domain['batch'][train_idx])&set(domain['batch'][val_idx]))==0
assert len(set(domain['batch'][train_idx])&set(domain['batch'][test_idx]))==0
assert len(set(domain['batch'][val_idx])&set(domain['batch'][test_idx]))==0
assert all(np.all(np.bincount(domain['y'][ix],minlength=4)>0) for ix in sets.values())

stress=buat_dataset(np.arange(100,106),SEED+802,'B',stress=True)
print('Domain stress:',stress['images'].shape,'perangkat belum terlihat:',stress['device'])

## 8. Ekstraksi fitur intensitas dan chromaticity

Rata-rata RGB pada daerah objek merangkum warna dominan tetapi mempertahankan pengaruh skala iluminasi. Untuk vektor RGB linear $\mathbf c=(R,G,B)$ dengan total intensitas $S=R+G+B$, *chromaticity* didefinisikan

$$
r=\frac{R}{S},\qquad g=\frac{G}{S},\qquad b=\frac{B}{S},\qquad r+g+b=1.
$$

Karena hanya dua komponen independen, kita menggunakan $(r,g)$ sebagai fitur. Perkalian **skalar yang sama** pada tiga kanal mempertahankan chromaticity; tetapi perkalian kanal berbeda atau pencampuran antar-kanal tidak mempertahankannya. Ketika $S\rightarrow0$, rasio ini menjadi sangat peka terhadap noise. Sebab itu kita sertakan $\varepsilon$ kecil hanya untuk mencegah pembagian nol numerik, bukan untuk menghilangkan masalah ilmiah pada intensitas rendah.

Rata-rata diambil dari area kiri, sedangkan *patch* netral dari kanan. Contoh ini memakai ROI yang diketahui persis karena gambar dihasilkan oleh simulator; pada kamera nyata, estimasi ROI memerlukan registrasi atau segmentasi dan menjadi sumber ketidakpastian tambahan.

In [ ]:
def ambil_rgb(images):
    images=np.asarray(images)
    w=images.shape[2]
    return images[:,:,:w//2,:].mean(axis=(1,2)),images[:,:,w//2:,:].mean(axis=(1,2))

def chromaticity(rgb,eps=1e-8):
    z=np.asarray(rgb,dtype=float)
    denom=np.sum(z,axis=-1,keepdims=True)
    return z[...,:2]/np.maximum(denom,eps)

rgb_obj,rgb_neutral=ambil_rgb(domain['images'])
stress_obj,stress_neutral=ambil_rgb(stress['images'])
fig,ax=plt.subplots(1,2,figsize=(11,4.1))
sc=ax[0].scatter(rgb_obj[:,0],rgb_obj[:,1],c=domain['y'],cmap='viridis',s=10,alpha=.55)
ax[0].set(xlabel='R sensor (linear sintetis)',ylabel='G sensor',title='Rerata RGB mentah');ax[0].grid(alpha=.2)
ch=chromaticity(rgb_obj)
ax[1].scatter(ch[:,0],ch[:,1],c=domain['y'],cmap='viridis',s=10,alpha=.55)
ax[1].set(xlabel='Chromaticity r',ylabel='Chromaticity g',title='Normalisasi intensitas total')
ax[1].grid(alpha=.2);fig.colorbar(sc,ax=ax.ravel().tolist(),label='Indeks kelas',shrink=.76)
plt.show()

## 9. White balance diagonal: estimasi, derivasi, dan batasnya

Apabila target patch netral diketahui memiliki nilai linear $n_0$ yang sama untuk tiga kanal, sementara sensor mengukur $\mathbf n=(n_R,n_G,n_B)$, koreksi diagonal berbasis pengamatan dapat ditulis

$$
\widehat{\mathbf r}_{\mathrm{WB}}=\operatorname{diag}\!\left(\frac{n_0}{n_R},\frac{n_0}{n_G},\frac{n_0}{n_B}\right)\mathbf y.
$$

Koreksi ini menghapus faktor gain kanal jika model ideal memang diagonal, offset nol, tidak ada clipping, dan patch netral menerima pencahayaan yang sama dengan sampel. Jika sensitivitas warna perangkat berubah melalui matriks non-diagonal $\mathbf M_d$, referensi netral tidak menentukan seluruh respons spektral. Tambahan masalah muncul karena pembagi $n_k$ sendiri bising. Pada data ini patch netral tidak berada di posisi yang sama persis dengan objek sehingga terdapat sisa *shading*.

**Perhatian metrologi:** faktor $n_0$ diset karena kita mengetahui target sintetis. Pada kamera nyata, nilai yang setara harus diputuskan melalui karakterisasi sistem; kartu abu-abu yang dipotret sebagai JPEG tidak otomatis mengembalikan RGB linear secara fisik.

In [ ]:
def white_balance(obj,ref,neutral=NEUTRAL,eps=.035):
    obj=np.asarray(obj,dtype=float)
    ref=np.asarray(ref,dtype=float)
    return obj*(neutral/np.maximum(ref,eps))

wb_obj=white_balance(rgb_obj,rgb_neutral)
wb_stress=white_balance(stress_obj,stress_neutral)
fig,ax=plt.subplots(1,2,figsize=(10.8,3.8))
for j,lab in enumerate(NAMA_KELAS):
    sel=domain['y']==j
    ax[0].scatter(rgb_obj[sel,0],rgb_obj[sel,1],alpha=.28,s=13,label=lab)
    ax[1].scatter(wb_obj[sel,0],wb_obj[sel,1],alpha=.28,s=13,label=lab)
ax[0].set(title='Sebelum WB',xlabel='R',ylabel='G');ax[1].set(title='Sesudah WB',xlabel='R terkoreksi',ylabel='G terkoreksi')
for a in ax:a.grid(alpha=.2)
ax[1].legend(loc='best',fontsize=8);plt.tight_layout();plt.show()

## 10. Mengapa ketidakpastian patch netral tidak boleh diabaikan?

Untuk satu kanal, $z=xn_0/n$ dengan pengukuran $x$ dan $n$ serta nilai referensi tetap $n_0$. Pendekatan propagasi ketidakpastian orde pertama memberi

$$
\operatorname{Var}(z)\simeq\left(\frac{n_0}{n}\right)^2\operatorname{Var}(x)
+\left(\frac{xn_0}{n^2}\right)^2\operatorname{Var}(n)
-2\frac{xn_0^2}{n^3}\operatorname{Cov}(x,n).
$$

Turunan $\partial z/\partial n=-xn_0/n^2$ menjelaskan mengapa noise meningkat saat sinyal netral mendekati nol. Rumus ini berlaku untuk noise relatif kecil, fungsi cukup mulus, dan tidak ada pembatasan clipping yang mendominasi. Asumsi independensi $x,n$ **bukan** selalu benar karena keduanya dapat terpengaruh fluktuasi paparan bersama. Kita bandingkan prediksi metode delta dengan Monte Carlo pada kondisi yang terkontrol.

In [ ]:
MC=np.random.default_rng(SEED+3)
num=45000
xmu=.35;nmu=.42; sx=.014;sn=.019;n0=.42
xsim=MC.normal(xmu,sx,num);nsim=MC.normal(nmu,sn,num)
zsim=xsim*n0/nsim
predvar=(n0/nmu)**2*sx**2+(xmu*n0/nmu**2)**2*sn**2
fig,ax=plt.subplots(figsize=(8.8,3.5))
ax.hist(zsim,bins=65,density=True,alpha=.7,label='Monte Carlo (noise independen)')
ax.axvline(zsim.mean(),color='black',lw=1,label='rerata Monte Carlo')
ax.set(xlabel='Kanal terkoreksi z',ylabel='Densitas',title='Propagasi noise pada white balance')
ax.legend();plt.show()
print('Simpangan baku (MC):',round(float(zsim.std(ddof=1)),5),
      ' | linearisasi:',round(float(np.sqrt(predvar)),5))
assert abs(zsim.std(ddof=1)-np.sqrt(predvar))/np.sqrt(predvar)<.05

## 11. Color correction matrix: estimasi multioutput dari chart pelatihan

Sebuah matriks koreksi linier menghubungkan nilai sensor yang telah dikoreksi WB ke nilai target linear referensi:

$$
\widehat{\mathbf r}=\mathbf A\mathbf x+\mathbf b,
\quad \mathbf x=\widehat{\mathbf r}_{\rm WB}.
$$

Kita menggunakan model afine dengan tiga keluaran. Jika $\mathbf X\in\mathbb R^{N\times4}$ memuat tiga kanal dan kolom satu, sedangkan $\mathbf Y\in\mathbb R^{N\times3}$ memuat target referensi, estimasi ridge multioutput diperoleh dengan meminimalkan

$$
J(\mathbf W)=\|\mathbf X\mathbf W-\mathbf Y\|_F^2+\lambda\|\mathbf D\mathbf W\|_F^2,
$$

sehingga

$$
\widehat{\mathbf W}=(\mathbf X^\top\mathbf X+\lambda\mathbf D^\top\mathbf D)^{-1}\mathbf X^\top\mathbf Y.
$$

$\mathbf D=\operatorname{diag}(1,1,1,0)$ menghindari penalti terhadap intersep. Dalam implementasi kita menggunakan `np.linalg.solve` dan bukan membentuk invers eksplisit. Karena pengamatan chart memiliki variasi dan noise, regularisasi dapat mengurangi ketidakstabilan numerik. Kalibrasi ini menggunakan **chart dengan warna target yang diketahui**; parameter fit **tidak boleh dipelajari dari chart test**.

Metode ini belum menggambarkan LUT 3D, profil ICC, transformasi spektral, adaptasi kromatik yang lengkap, maupun pengelolaan gamut. Ia hanya baseline terkontrol untuk memperlihatkan pentingnya kalibrasi terpisah.

In [ ]:
def chart_pada_split(dataset,batches):
    mask=np.isin(dataset['chart_batch'],np.asarray(batches))
    x=white_balance(dataset['chart_obs'][mask],dataset['chart_ref'][mask])
    y=dataset['chart_target'][mask]
    return x,y

def fit_ccm(x,y,lam=.01):
    X=np.column_stack([np.asarray(x),np.ones(len(x))]);Y=np.asarray(y)
    D=np.diag([1.,1.,1.,0.])
    return np.linalg.solve(X.T@X+lam*D,X.T@Y)

def apply_ccm(x,W,clip=True):
    val=np.column_stack([np.asarray(x),np.ones(len(x))])@W
    return np.clip(val,0,1) if clip else val

def lab_from_linear(x):
    col=np.asarray(x,dtype=float)
    return rgb2lab(srgb_encode(np.clip(col,0,1)).reshape(-1,1,3)).reshape(-1,3)

def deltae_per_sample(x,y):
    lab1=lab_from_linear(x);lab2=lab_from_linear(y)
    return deltaE_ciede2000(lab1[:,None,:],lab2[:,None,:]).ravel()

train_batches=np.unique(domain['batch'][train_idx]);val_batches=np.unique(domain['batch'][val_idx])
Xt,Yt=chart_pada_split(domain,train_batches)
Xv,Yv=chart_pada_split(domain,val_batches)
print('Pasangan chart train:',Xt.shape,'| validation:',Xv.shape,
      '| kondisi X:',round(np.linalg.cond(np.column_stack([Xt,np.ones(len(Xt))])),2))

## 12. Pemilihan regularisasi menggunakan chart validation

Parameter $\lambda$ pada CCM dipilih menggunakan warna chart dari batch **validation**. Evaluasi tidak boleh berdasarkan seberapa baik model cocok dengan titik train saja: dengan $\lambda$ mendekati nol, residual pelatihan dapat kecil namun sensitivitas terhadap noise meningkat. Kita tampilkan dua metrik validation: rata-rata galat kontinu pada RGB linear dan rata-rata beda warna terhadap referensi sintetis. Nilai $\Delta E_{00}$ dihitung setelah konversi referensi linear sRGB ke Lab; interpretasi kolorimetrik kamera nyata **tidak** diasumsikan.

In [ ]:
calrows=[];CCMS={}
for lam in [0.,1e-4,1e-3,1e-2,.1,1.,10.]:
    W=fit_ccm(Xt,Yt,lam)
    Cval=apply_ccm(Xv,W)
    calrows.append({'lambda':lam,'RMSE val RGB':float(np.sqrt(np.mean((Cval-Yv)**2))),
                    'DeltaE00 val (proxy)':float(np.mean(deltae_per_sample(Cval,Yv)))})
    CCMS[lam]=W
calibration_table=pd.DataFrame(calrows)
print(calibration_table.round(4).to_string(index=False))
chosen_lam=float(calibration_table.sort_values(['DeltaE00 val (proxy)','lambda']).iloc[0]['lambda'])
Wccm=CCMS[chosen_lam]
print('Lambda terpilih dari chart validation:',chosen_lam,'| CCM:\n',np.round(Wccm,3))
fig,ax=plt.subplots(figsize=(8.5,3.8))
ax.semilogx(calibration_table['lambda'].replace(0,1e-6),calibration_table['DeltaE00 val (proxy)'],'o-')
ax.axvline(max(chosen_lam,1e-6),ls='--',color='black',label='dipilih dari validasi')
ax.set(xlabel='Regularisasi lambda (0 digambar pada 1e-6)',ylabel='Rerata DeltaE00 proxy',title='Validasi regularisasi CCM')
ax.grid(alpha=.25);ax.legend();plt.show()

## 13. Memeriksa akurasi warna tanpa menggunakan data test untuk tuning

Sesudah CCM terpilih, kita dapat membandingkan tiga keluaran: sensor mentah (sebagai *naive RGB proxy*), hasil koreksi diagonal (WB), dan hasil WB+CCM. Ketiganya dibandingkan terhadap referensi sintetis yang dibuat sebelum simulasi sensor.

Warna pada panel sensor mentah **bukan koordinat sRGB** sehingga $\Delta E_{00}$ yang dihitung dari panel mentah hanya relevan sebagai ukuran *proxy* dalam eksperimen, bukan klaim bahwa output RAW sensor dapat langsung dinilai menggunakan sRGB. Setelah koreksi, interpretasi lebih dekat dengan target referensi, tetapi tetap memerlukan karakterisasi fisik jika diaplikasikan pada perangkat nyata. Nilai rata-rata dan ekor distribusi perlu diperiksa bersama, tidak cukup hanya satu nilai rerata.

In [ ]:
def fitur(dataset,W):
    obj,ref=ambil_rgb(dataset['images'])
    wb=white_balance(obj,ref)
    ccm=apply_ccm(wb,W)
    return {'RGB mentah':obj,'Chromaticity':chromaticity(obj),
            'White balance':wb,'WB + CCM':ccm,
            'Lab dari CCM':lab_from_linear(ccm)}, {'ref':ref,'wb':wb,'ccm':ccm,'raw':obj}

FIT,RAW=fitur(domain,Wccm)
FIT_STRESS,RAW_STRESS=fitur(stress,Wccm)
for name,arr in [('Mentah (proxy)',RAW['raw']),('WB',RAW['wb']),('WB+CCM',RAW['ccm'])]:
    delta=deltae_per_sample(arr[test_idx],domain['target'][test_idx])
    print(f'{name:16s} DeltaE00 proxy test: mean={delta.mean():.3f}, median={np.median(delta):.3f}, P90={np.percentile(delta,90):.3f}')

fig,axs=plt.subplots(1,2,figsize=(11.6,4.1))
for label,name,key in [('Mentah (proxy)','Mentah (proxy)','raw'),('WB','WB','wb'),('WB+CCM','WB+CCM','ccm')]:
    delta=deltae_per_sample(RAW[key][test_idx],domain['target'][test_idx])
    axs[0].hist(delta,bins=np.linspace(0,max(24,delta.max()),28),alpha=.45,label=label)
    axs[1].plot(np.sort(delta),np.linspace(0,1,len(delta)),label=label)
axs[0].set(xlabel='DeltaE00 proxy',ylabel='Frekuensi',title='Distribusi beda warna pada test')
axs[1].set(xlabel='DeltaE00 proxy',ylabel='CDF empirik',title='Galat ekor juga penting')
for ax in axs:ax.legend(fontsize=8);ax.grid(alpha=.2)
plt.tight_layout();plt.show()

## 14. Pengukuran warna dan klasifikasi adalah dua objektif berbeda

Untuk klasifikasi multikelas dengan fitur $\mathbf x$, model regresi logistik multinomial membentuk *logit* $z_k=\mathbf w_k^\top\mathbf x+b_k$ dan probabilitas

$$
p_k(\mathbf x)=\frac{\exp(z_k)}{\sum_{j=1}^K\exp(z_j)}.
$$

Estimasi parameter dapat diperoleh dengan meminimalkan *cross-entropy* teratur:

$$
\mathcal L(\theta)=-\sum_{i=1}^N\log p_{y_i}(\mathbf x_i)+\alpha\|\mathbf W\|_F^2.
$$

Fungsi biaya ini mengoptimalkan prediksi label, bukan beda warna CIE. Pengukuran $\Delta E_{00}$ dan prediksi probabilitas kelas memerlukan *ground truth* yang berbeda. Pada notebook ini, semua fitur boleh diuji pada validation, tetapi **aturan pemilihan akhir ditetapkan dari validation saja**. Model pembanding menggunakan regresi logistik untuk semua variasi fitur dan SVM RBF sebagai pembanding nonlinier untuk fitur WB/CCM; *StandardScaler* dipasang di dalam pipeline sehingga tidak pernah belajar dari seluruh dataset.

In [ ]:
def create_model(alg,C):
    if alg=='LogReg':
        estimator=LogisticRegression(C=C,max_iter=2500,random_state=SEED)
    else:
        estimator=SVC(C=C,kernel='rbf',gamma='scale',probability=True,random_state=SEED)
    return Pipeline([('scale',StandardScaler()),('model',estimator)])

experiments=[]
feature_order=list(FIT.keys())
for name in feature_order:
    algorithms=['LogReg','SVM-RBF'] if name in ['White balance','WB + CCM'] else ['LogReg']
    for alg in algorithms:
        for Cpar in [.1,1,10]:
            mdl=create_model(alg,Cpar)
            mdl.fit(FIT[name][train_idx],domain['y'][train_idx])
            pred=mdl.predict(FIT[name][val_idx])
            experiments.append({'fitur':name,'model':alg,'C':Cpar,
                    'F1 validasi':float(f1_score(domain['y'][val_idx],pred,average='macro')),
                    'Akurasi validasi':float(accuracy_score(domain['y'][val_idx],pred))})
val_table=pd.DataFrame(experiments)
print(val_table.sort_values('F1 validasi',ascending=False).head(14).round(3).to_string(index=False))

## 15. Pemilihan model ditetapkan sebelum membuka data test

Pilih kombinasi fitur, model, dan regularisasi dari tabel validation. Untuk nilai validasi yang sama (jika terjadi), kita tetapkan aturan *tie-break* sederhana terlebih dahulu: preferensi model linear dan kemudian parameter $C$ lebih kecil. Aturan ini mencerminkan kehati-hatian terhadap kompleksitas yang tidak memberi bukti keuntungan. **Test tidak dipakai untuk mengganti model setelah melihat hasilnya.** Kita juga menyimpan satu model pembanding RGB mentah agar pengaruh koreksi dapat diperiksa secara berpasangan.

In [ ]:
val_rank=val_table.copy()
val_rank['kompleksitas']=val_rank['model'].map({'LogReg':0,'SVM-RBF':1})
val_rank['fitur_prioritas']=val_rank['fitur'].map({
    'RGB mentah':3,'Chromaticity':2,'White balance':1,'WB + CCM':0,'Lab dari CCM':0})
# Tie-break sudah dideklarasikan dalam materi sebelum evaluasi test.
val_rank=val_rank.sort_values(['F1 validasi','kompleksitas','C','fitur_prioritas'],ascending=[False,True,True,True])
best=val_rank.iloc[0]
print('MODEL TERPILIH BERDASARKAN VALIDASI:',best[['fitur','model','C','F1 validasi']].to_dict())
selected=create_model(best['model'],float(best['C']))
selected.fit(FIT[best['fitur']][train_idx],domain['y'][train_idx])
raw_baseline=create_model('LogReg',1)
raw_baseline.fit(FIT['RGB mentah'][train_idx],domain['y'][train_idx])
print('Aturan seleksi sudah final: tidak ada tuning berdasarkan test.')

## 16. Uji akhir: membandingkan fitur pada data test yang sama

Walaupun hanya satu konfigurasi menjadi *model terpilih*, kita juga menampilkan beberapa baseline yang sudah didefinisikan sebelumnya untuk diagnosis ilmiah. Nilai pada test bersifat **evaluasi akhir**, tidak boleh dijadikan alasan untuk memilih ulang model. Macro-F1 menghitung rerata F1 keempat kelas sehingga kelas kecil tidak otomatis tertutupi oleh kelas besar. Karena dataset seimbang, akurasi berguna sebagai pelengkap, tetapi tetap tidak membahas kesalahan probabilitas atau kualitas warna.

In [ ]:
# Baseline diagnostik dipra-tetapkan: LogReg C=1 pada masing-masing fitur.
models_diag={}
rows=[]
for name in feature_order:
    model=create_model('LogReg',1.0)
    model.fit(FIT[name][train_idx],domain['y'][train_idx])
    pp=model.predict(FIT[name][test_idx])
    ps=model.predict(FIT_STRESS[name])
    models_diag[name]=model
    rows.append({'Fitur':name,'F1 test utama':f1_score(domain['y'][test_idx],pp,average='macro'),
                 'F1 stress':f1_score(stress['y'],ps,average='macro'),
                 'Akurasi test':accuracy_score(domain['y'][test_idx],pp)})
results=pd.DataFrame(rows)
print(results.round(3).to_string(index=False))

p_final=selected.predict(FIT[best['fitur']][test_idx]);
p_stress=selected.predict(FIT_STRESS[best['fitur']])
summary_selected={
    'F1 test':float(f1_score(domain['y'][test_idx],p_final,average='macro')),
    'F1 stress':float(f1_score(stress['y'],p_stress,average='macro')),
    'Akurasi test':float(accuracy_score(domain['y'][test_idx],p_final))}
print('Model terpilih:',summary_selected)

## 17. Mengapa perbaikan warna tidak selalu berarti perbaikan klasifikasi?

Korelasi antara beda warna dan keberhasilan klasifikasi bukan identitas matematis. Koreksi kolorimetri meminimalkan ketidaksesuaian koordinat warna terhadap referensi; classifier meminimalkan kesalahan label pada batas keputusan yang dipelajari. Warna yang lebih dekat ke referensi tidak selalu menggeser titik ke sisi batas keputusan yang benar, terutama bila sebaran kelas saling tumpang tindih atau model telah belajar toleransi tertentu terhadap perubahan iluminasi. Karena itu perbedaan pada grafik sebelumnya harus dibaca **apa adanya**. Hindari memilih metode koreksi berdasarkan F1 test, sebab prosedur itu menyebabkan *model selection leakage*.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(12,4.2))
x=np.arange(len(results));w=.36
ax[0].bar(x-w/2,results['F1 test utama'],w,label='Test utama')
ax[0].bar(x+w/2,results['F1 stress'],w,label='Perangkat/iluminan stress')
ax[0].set_xticks(x,results['Fitur'],rotation=28,ha='right')
ax[0].set(ylabel='Macro-F1',ylim=(0,1.06),title='Generalization gap menurut fitur')
ax[0].legend();ax[0].grid(axis='y',alpha=.2)
ax[1].scatter(results['F1 test utama'],results['F1 stress'],s=70)
for i,r in results.iterrows():
    ax[1].annotate(r['Fitur'],(r['F1 test utama'],r['F1 stress']),xytext=(4,5),textcoords='offset points',fontsize=8)
ax[1].plot([0,1],[0,1],'--',alpha=.5)
ax[1].set(xlim=(0,1.06),ylim=(0,1.06),xlabel='F1 test utama',ylabel='F1 domain stress',title='F1 stress tidak dijamin sama')
ax[1].grid(alpha=.2);plt.tight_layout();plt.show()

## 18. Confusion matrix dan diagnosis kesalahan antarkelas

Macro-F1 merangkum kinerja, tetapi tidak memberitahu kelas mana yang saling tertukar. Pada aplikasi pengukuran, dua label mungkin berdekatan secara kolorimetrik sehingga diperlukan biaya kesalahan yang berbeda. *Confusion matrix* dihitung pada model terpilih dan test yang belum dipakai selama seleksi. Kita tampilkan normalisasi per kelas benar untuk melihat apakah suatu kelas memiliki *recall* rendah meskipun akurasi keseluruhan tinggi.

In [ ]:
fig,ax=plt.subplots(1,2,figsize=(11,4.4))
ConfusionMatrixDisplay.from_predictions(domain['y'][test_idx],p_final,
    labels=np.arange(4),display_labels=['A','B','C','D'],normalize='true',ax=ax[0],cmap='Blues',values_format='.2f')
ax[0].set_title('Model terpilih — test utama')
ConfusionMatrixDisplay.from_predictions(stress['y'],p_stress,
    labels=np.arange(4),display_labels=['A','B','C','D'],normalize='true',ax=ax[1],cmap='Blues',values_format='.2f')
ax[1].set_title('Model terpilih — domain stress')
plt.tight_layout();plt.show()
print('Classification report (test utama):\n',classification_report(domain['y'][test_idx],p_final,labels=np.arange(4),target_names=['A','B','C','D'],zero_division=0))

## 19. Kalibrasi probabilitas: akurasi saja belum cukup

Probabilitas keluaran klasifikator $\hat p$ belum tentu merepresentasikan frekuensi kejadian yang benar. Suatu model yang mengeluarkan probabilitas 0,9 untuk banyak prediksi idealnya benar sekitar 90% dalam kelompok tersebut, bila probabilitasnya terkalibrasi. Untuk klasifikasi multikelas kita gunakan *top-label expected calibration error* (ECE) sebagai indikator ringkas:

$$
\operatorname{ECE}=\sum_{m=1}^M \frac{|B_m|}{N}
\big|\operatorname{acc}(B_m)-\operatorname{conf}(B_m)\big|.
$$

ECE sensitif terhadap jumlah bin dan ukuran sampel, sehingga bukan bukti kalibrasi sempurna. Kita juga menghitung Brier multikelas sebagai rerata jumlah kuadrat galat antara probabilitas dan indikator kelas benar. **Kalibrasi ulang** yang memerlukan data validation tambahan (misalnya temperature scaling) dapat dilakukan, tetapi parameter kalibrasi tidak boleh dipelajari dari test.

In [ ]:
def evaluasi_kalibrasi(p,y,n_bins=8):
    p=np.asarray(p);y=np.asarray(y)
    conf=p.max(axis=1); pred=p.argmax(axis=1);hit=(pred==y).astype(float)
    edges=np.linspace(0,1,n_bins+1)
    bins=np.minimum(np.digitize(conf,edges[1:-1]),n_bins-1)
    centers=[];acc=[];confidence=[];weights=[]
    for i in range(n_bins):
        mask=bins==i
        if mask.any():
            centers.append((edges[i]+edges[i+1])/2);acc.append(hit[mask].mean())
            confidence.append(conf[mask].mean());weights.append(mask.mean())
    brier=np.mean(np.sum((p-np.eye(p.shape[1])[y])**2,axis=1))
    return {'ECE':sum(w*abs(a-c) for w,a,c in zip(weights,acc,confidence)),
            'Brier':brier,'confidence':np.array(confidence),'accuracy':np.array(acc),
            'weight':np.array(weights),'centers':np.array(centers)}

p_prob=selected.predict_proba(FIT[best['fitur']][test_idx])
cal=evaluasi_kalibrasi(p_prob,domain['y'][test_idx])
print('ECE top-label:',round(cal['ECE'],4),'Brier multiclass:',round(cal['Brier'],4))
fig,ax=plt.subplots(figsize=(6.5,4.5))
ax.plot([0,1],[0,1],'--',label='ideal')
ax.scatter(cal['confidence'],cal['accuracy'],s=np.maximum(40,cal['weight']*450),label='test utama')
ax.set(xlim=(0,1),ylim=(0,1),xlabel='Confidence rata-rata',ylabel='Akurasi empirik',title='Reliability diagram (model terpilih)')
ax.legend();ax.grid(alpha=.25);plt.show()

## 20. Uji perangkat tak terlihat: apa yang termasuk domain shift?

Data stress menggunakan matriks respons perangkat B dan variasi gain iluminan lebih kuat, bukan sekadar menambahkan noise Gaussian ke citra test yang sama. Dengan demikian, model menghadapi **pergeseran data input yang memiliki penyebab fisik terkontrol**. Domain stress tetap buatan; di laboratorium nyata, pergeseran dapat berasal dari LED dengan SPD berbeda, lensa, firmware, pipeline JPEG, profil warna, paparan otomatis, posisi kamera, atau komposisi material yang berubah.

Pada *device shift*, WB diagonal belum tentu memulihkan pemetaan jika matriks pencampuran kanal berubah. CCM yang dilatih untuk perangkat A juga tidak otomatis berlaku pada perangkat B; memperbarui CCM dengan chart dari perangkat B akan menjadi eksperimen **adaptasi domain**, berbeda dengan generalisasi tanpa data label/perangkat baru. Kita tidak melakukan adaptasi tersebut di sini.

In [ ]:
ref_test=domain['target'][test_idx]
ref_stress=stress['target']
fig,ax=plt.subplots(1,2,figsize=(11.3,4.1))
for j,(data,target,title) in enumerate([(RAW,ref_test,'Test perangkat A'),(RAW_STRESS,ref_stress,'Stress perangkat B')]):
    for label,key in [('Mentah (proxy)','raw'),('WB','wb'),('WB+CCM','ccm')]:
        subset=data[key][test_idx] if j==0 else data[key]
        d=deltae_per_sample(subset,target)
        ax[j].boxplot(d,positions=[['Mentah (proxy)','WB','WB+CCM'].index(label)+1],widths=.45,
                      patch_artist=True,boxprops={'facecolor':'lightgray'},showfliers=False)
    ax[j].set_xticks([1,2,3],['Mentah','WB','WB+CCM'])
    ax[j].set(ylabel='DeltaE00 proxy',title=title)
    ax[j].grid(axis='y',alpha=.2)
plt.tight_layout();plt.show()

## 21. Ketidakpastian per batch: paired cluster bootstrap

Semua frame dari satu batch dipengaruhi iluminan bersama. Jika kita melakukan bootstrap per frame, estimasi ketidakpastian dapat terlalu kecil. Untuk membandingkan dua metode pada **data test yang sama**, kita resampling **batch** dengan pengembalian, lalu menghitung ulang macro-F1 pada gabungan frame dari batch terpilih. Selisih dalam tiap sampel bootstrap dihitung berpasangan, sehingga perbedaan antar-metode tidak tercampur dengan perubahan sampel.

Dengan hanya enam batch test, hasil interval ini masih kasar dan tidak mewakili variasi seluruh kamera atau material. Interval 95% bootstrap hanyalah estimasi ketidakpastian sampling **dengan kondisi model tetap**; belum memasukkan pemilihan model, pelatihan ulang, atau ketidakpastian kalibrasi fisik.

In [ ]:
def bootstrap_paired_batches(y,p1,p2,groups,n_boot=700,seed=12345):
    r=np.random.default_rng(seed)
    unique=np.unique(groups);n=len(unique);diff=[];m1=[];m2=[]
    for _ in range(n_boot):
        selected_groups=r.choice(unique,size=n,replace=True)
        ix=np.concatenate([np.flatnonzero(groups==b) for b in selected_groups])
        s1=f1_score(y[ix],p1[ix],average='macro',labels=np.arange(4),zero_division=0)
        s2=f1_score(y[ix],p2[ix],average='macro',labels=np.arange(4),zero_division=0)
        diff.append(s1-s2);m1.append(s1);m2.append(s2)
    return np.asarray(diff),np.asarray(m1),np.asarray(m2)

raw_test=models_diag['RGB mentah'].predict(FIT['RGB mentah'][test_idx])
bt,aa,bb=bootstrap_paired_batches(domain['y'][test_idx],p_final,raw_test,domain['batch'][test_idx])
ci=np.percentile(bt,[2.5,97.5]);observed=(f1_score(domain['y'][test_idx],p_final,average='macro')-
    f1_score(domain['y'][test_idx],raw_test,average='macro'))
print(f'Selisih F1 (terpilih - RGB mentah): {observed:+.3f}, CI bootstrap batch 95% [{ci[0]:+.3f}, {ci[1]:+.3f}]')
fig,ax=plt.subplots(figsize=(8.2,3.6))
ax.hist(bt,bins=27,alpha=.8);ax.axvline(observed,color='black',lw=1.3,label='selisih observasi')
ax.axvline(0,color='gray',ls='--');ax.set(xlabel='Selisih macro-F1',ylabel='Frekuensi bootstrap',
    title='Bootstrap berpasangan pada unit batch test (n=6)')
ax.legend();plt.tight_layout();plt.show()

## 22. Ablation: referensi netral, transformasi, dan kompleksitas model

*Ablation study* harus menjawab pertanyaan kausal terbatas: apakah komponen tertentu memberikan kontribusi pada konfigurasi eksperimen yang dikontrol? Kita memiliki tiga keputusan desain yang dapat dibandingkan dengan fitur dan pipeline sama: tanpa referensi netral (RGB mentah), dengan normalisasi kanal (WB), serta dengan transformasi chart multiwarna (WB+CCM). Perbandingan dilakukan dengan model linear yang sama, regularisasi tetap, split tetap, dan metrik sama.

Bila WB+CCM tidak unggul, jangan menyimpulkan koreksi warna tidak berguna secara umum: bisa jadi pemilihan warna, kelas, domain, atau noise membuat manfaatnya tidak terlihat oleh classifier. Jika WB+CCM unggul pada $\Delta E_{00}$ tetapi bukan F1, hal itu pun sah dan konsisten dengan perbedaan fungsi objektif.

In [ ]:
abl=results[results['Fitur'].isin(['RGB mentah','White balance','WB + CCM'])].copy()
abl['Gap F1']=abl['F1 test utama']-abl['F1 stress']
print('ABLATION (LogReg C=1, split yang sama):\n',abl.round(3).to_string(index=False))
fig,ax=plt.subplots(figsize=(8.5,3.6))
for _,r in abl.iterrows():
    ax.plot([0,1],[r['F1 test utama'],r['F1 stress']],'o-',label=r['Fitur'],lw=2)
ax.set(xticks=[0,1],xticklabels=['Test utama','Domain stress'],ylabel='Macro-F1',ylim=(0,1.04),
    title='Komponen pipeline dan sensitivitas terhadap pergeseran domain')
ax.grid(alpha=.2);ax.legend();plt.tight_layout();plt.show()

## 23. Kesalahan umum dan ancaman terhadap validitas

**(a) Kebocoran data.** Memisah frame satu per satu ketika frame berbagi sesi atau identitas benda yang sama menyebabkan ketergantungan train dan test. Memilih parameter CCM berdasarkan chart test juga membuat test tidak independen.

**(b) Kalibrasi yang terlalu dinyatakan.** Mengalikan gain kanal bukan karakterisasi spektral, dan matriks koreksi warna sintetis bukan jaminan akurasi perangkat nyata. Istilah $\Delta E_{00}$ harus selalu menyebut referensi warna, ruang warna, iluminan dan pipeline konversinya.

**(c) Ketidakpastian yang diabaikan.** Patch abu-abu dapat bising, tidak seragam, terlalu gelap, atau mengalami saturasi. Nilai pembagi kecil memperbesar noise. Model tidak bisa mengidentifikasi informasi spektral yang memang tidak terukur oleh sensor tiga kanal.

**(d) Generalisasi yang dibesar-besarkan.** Unit batch, representasi perangkat, variasi illuminan, dan cara label dibuat menentukan batas klaim. Pada data sintetis, kita tahu faktor pembangkit; kamera nyata menambahkan perubahan firmware, penampang respons, geometri, dan material yang belum tentu tercakup.

**(e) Peringkat dari satu angka.** Kinerja klasifikasi, beda warna, kurva kalibrasi probabilitas, dan durasi komputasi menjawab pertanyaan berbeda. Penilaian ilmiah harus menyajikan beberapa indikator relevan tanpa memaksakan satu metode sebagai pemenang universal.

## 24. Audit keterulangan: kontrak data, model, dan hasil

Audit berikut memeriksa ukuran dataset, isolasi batch, dimensi fitur, nilai hingga (*finite*), rekonstruksi fungsi transfer sRGB, dan konsistensi probabilitas. Audit otomatis membantu mendeteksi kesalahan implementasi, tetapi **tidak** membuktikan validitas eksternal pada perangkat fisik. Simpan versi pustaka, parameter, seed, serta aturan split ketika mereproduksi atau membandingkan eksperimen.

In [ ]:
assert domain['images'].shape==(720,24,48,3)
assert stress['images'].shape==(120,24,48,3)
assert len(np.unique(domain['batch']))==36
assert len(np.unique(stress['batch']))==6
assert len(np.unique(domain['batch'][train_idx]))==24
assert len(np.unique(domain['batch'][val_idx]))==6
assert len(np.unique(domain['batch'][test_idx]))==6
assert set(domain['batch'][train_idx]).isdisjoint(domain['batch'][test_idx])
assert set(domain['batch'][val_idx]).isdisjoint(domain['batch'][test_idx])
assert np.all((domain['images']>=0)&(domain['images']<=1))
assert np.all(np.isfinite(domain['images']))
assert np.all(np.isfinite(Wccm))
assert all(np.all(np.isfinite(x)) for x in FIT.values())
assert np.allclose(p_prob.sum(axis=1),1,atol=1e-8)
assert np.all(np.isfinite(p_prob))
assert np.isfinite(cal['ECE']) and np.isfinite(cal['Brier'])
print('Pemeriksaan konsistensi data, model, dan split berhasil. Ini bukan sertifikat kalibrasi fisik.')

## 25. Ekspor dataset dan provenance sintetis

Untuk latihan mandiri, dataset utama dan domain stress disimpan dalam satu berkas NPZ pada folder keluaran lokal. File ini mengandung citra sintetis, label, batch, target linear referensi, pengamatan chart, dan pembagian indeks. Data dapat dipakai tanpa menjalankan ulang generator, tetapi manifest tetap harus mencatat **bahwa data dihasilkan oleh simulator**, seed, skala, bentuk array, serta asumsi sensor. Notebook bisa dieksekusi langsung tanpa berkas NPZ karena data dibangkitkan dari kode di atas. Jika memakai Google Colab, unduh NPZ dari panel berkas atau simpan ke Drive sesuai kebutuhan.

In [ ]:
dataset_path=OUTPUT_DIR/'11_ml_kolorimetri_dataset_sintetis.npz'
np.savez_compressed(dataset_path,
    images=domain['images'],y=domain['y'],batch=domain['batch'],target_linear_rgb=domain['target'],
    chart_obs=domain['chart_obs'],chart_neutral=domain['chart_ref'],chart_target=domain['chart_target'],
    chart_batch=domain['chart_batch'],train_idx=train_idx,val_idx=val_idx,test_idx=test_idx,
    stress_images=stress['images'],stress_y=stress['y'],stress_batch=stress['batch'],
    stress_target_linear_rgb=stress['target'],palette_linear_rgb=PALET,
    ccm=Wccm,ccm_lambda=np.array(chosen_lam),seed=np.array(SEED))
print('Dataset lokal tersimpan:',dataset_path,'| MB:',round(dataset_path.stat().st_size/1e6,2))

## 26. Latihan bertingkat dan pertanyaan HOTS

**Konsep dan derivasi**

1. Turunkan bagaimana perubahan skalar iluminan $a$ memengaruhi chromaticity ketika offset dan noise diabaikan. Lalu tunjukkan mengapa perubahan gain setiap kanal tidak memiliki sifat yang sama.
2. Gunakan model $\mathbf y=\operatorname{diag}(\boldsymbol\ell)\mathbf M\mathbf r+\mathbf b$ untuk menjelaskan mengapa satu pengukuran patch netral tidak cukup menentukan sembilan parameter $\mathbf M$.
3. Turunkan fungsi objektif dan *normal equations* untuk CCM dengan regularisasi ridge. Mengapa intersep tidak harus dipenalti?
4. Buktikan pendekatan propagasi varians pada koreksi $z=xn_0/n$ dan identifikasi kondisi kegagalannya.

**Eksperimen komputasional**

5. Naikkan `shift_scale` pada domain utama tanpa mengubah label; laporkan perubahan F1 untuk RGB mentah dan WB. Gunakan split batch yang sama ketika membandingkan metode.
6. Tambahkan offset kanal yang lebih besar pada simulator; periksa apakah WB diagonal menghasilkan bias yang dapat diprediksi secara matematis.
7. Kurangi intensitas patch netral sampai mendekati nol. Bandingkan galat teoritis metode delta dengan Monte Carlo dan jelaskan pengaruh clipping atau epsilon.
8. Lakukan *ablation* 3, 6, 12, dan 24 warna chart untuk CCM. Pastikan setiap konfigurasi menguji regularisasi pada batch validation yang sama.

**Penelitian dan penalaran tingkat tinggi**

9. Rancang protokol eksperimen nyata menggunakan dua smartphone, dua jenis sumber cahaya, chart referensi, dan spesimen yang sama. Tentukan unit eksperimen, jumlah pengulangan, pengendalian paparan, dan sumber *leakage*.
10. Susun evaluasi *leave-one-device-out*. Jelaskan kapan adaptasi dengan chart perangkat target masih boleh disebut generalisasi tanpa label dan kapan hasilnya harus dilaporkan sebagai adaptasi domain.
11. Bentuk keputusan dengan biaya salah klasifikasi tidak simetris. Jelaskan mengapa threshold optimal tidak selalu 0,5, dan bagaimana menjaga data test agar tidak ikut memilih threshold.
12. Buat *reproducibility card* yang menyatakan lisensi data, asal-usul target warna, versi library, random seed, unit bootstrap, metrik, batas klaim, serta hal-hal yang belum diuji.

**Tugas pelaporan:** tulis *mini technical report* 5–8 halaman yang berisi rumusan masalah; hipotesis; model pembentukan citra; desain split; algoritma dan fungsi objektif; hasil utama serta *domain stress*; interval ketidakpastian; kegagalan metode; dan rencana validasi perangkat nyata. Jangan menyajikan plot tanpa interpretasi.

## 27. Sintesis: apa yang sudah dan belum dapat disimpulkan?

Dari eksperimen ini kita dapat **mengamati** pengaruh fitur mentah, normalisasi chromaticity, koreksi diagonal, dan CCM teratur terhadap warna referensi sintetis dan klasifikasi empat kelas. Perbedaan pada data test utama dan domain stress dihasilkan oleh simulator yang diketahui; pemeriksaan batch terpisah mengurangi beberapa bentuk kebocoran. Kita dapat melaporkan nilai metrik yang benar-benar terhitung serta ketidakpastian *batch bootstrap* sesuai keterbatasannya.

Namun kita **belum dapat** menyimpulkan bahwa suatu metode telah terkalibrasi pada kamera nyata, bahwa nilai $\Delta E_{00}$ setara dengan pengukuran instrumen standar, ataupun bahwa satu model akan unggul pada material, operator, dan proses akuisisi lain. Model respons sensor masih merupakan aproksimasi tiga kanal dan unit eksperimen hanya batch sintetis.

**Koneksi berikutnya:** pembahasan machine learning dalam notebook 12 dapat diperluas menjadi rancangan studi kasus integratif dengan protokol pengumpulan data, perbandingan baseline, uji domain shift, serta pelaporan batas klaim. Bagian lanjutan tentang *inverse problems*, Bayesian imaging, metrologi dan reproducibility akan memperdalam hubungan antara respons sensor, ketidakpastian, dan keputusan model yang dapat diaudit.

## Referensi dan bacaan lanjutan

1. R. S. Berns. *Billmeyer and Saltzman's Principles of Color Technology*, edisi 3, Wiley, 2000 — dasar kolorimetri dan instrumentasi warna.
2. M. D. Fairchild. *Color Appearance Models*, edisi 3, Wiley, 2013 — perubahan kondisi pengamatan dan adaptasi warna.
3. G. Sharma, W. Wu, E. N. Dalal (2005). “The CIEDE2000 Color-Difference Formula: Implementation Notes, Supplementary Test Data, and Mathematical Observations.” *Color Research & Application*, 30(1), 21–30. DOI: 10.1002/col.20070.
4. ISO/CIE 11664-6:2014. *Colorimetry — Part 6: CIEDE2000 Colour-Difference Formula* — definisi standar beda warna.
5. M. Ebner. *Color Constancy*, Wiley, 2007 — iluminasi, perangkat, dan pendekatan koreksi warna.
6. C. M. Bishop. *Pattern Recognition and Machine Learning*, Springer, 2006 — fungsi kerugian probabilistik dan regularisasi.
7. Dokumentasi scikit-image: `skimage.color.rgb2lab` dan `deltaE_ciede2000`: https://scikit-image.org/docs/stable/api/skimage.color.html
8. Dokumentasi scikit-learn: `GroupShuffleSplit`, `Pipeline`, `LogisticRegression`, serta metrik klasifikasi: https://scikit-learn.org/stable/

**Catatan:** referensi ini dipakai untuk memperdalam prinsip. Dataset dan hasil pada notebook merupakan contoh original yang dihasilkan simulator; tidak ada data eksperimen nyata yang diambil dari referensi di atas.